In [0]:
from pyspark.sql import functions as f
import sys
sys.path.append('..')
sys.path.append('../..')

import lib_etl.validations_ETL as validations
from lib.job_manager import load_config, split_config

In [0]:
%run ../../config/utils

In [0]:
config = load_config(etl_config_path)
data_paths, club_square_config, config_validation = split_config(config)
run_as_date = dbutils.widgets.get("run_as_date")

### Transform 

In [0]:
df_payment = spark.sql(f"""
    SELECT 
        CAST(MBRSHP_SID AS LONG) AS MBRSHP_SID
        ,CAST(PURCH_HDR_ID AS LONG) AS PURCH_HDR_ID
        ,CAST(PURCH_PYMT_SEQ_ID AS INT) AS PURCH_PYMT_SEQ_ID
        ,TENDER_TYPE_CD
        ,CPN_NBR
        ,PYMT_SCANNED_OR_KEYED_IND
        ,CAST(SALES_PYMT_AMT AS DECIMAL(9,2)) AS SALES_PYMT_AMT
        ,TENDER_ID
    FROM 
        {bronze_transaction_payment}
""").dropDuplicates()

df_payment.createOrReplaceTempView("source")

In [0]:
validations.validate_table(
        spark, "intermediate", 'payment', config_validation, df_payment, stats_etl_path
    )

### Merge

In [0]:
df_payment.write.mode("overwrite").saveAsTable(silver_transaction_payment)

if archive_flag:
    save_archive(df_payment, silver_transaction_payment_archive, run_as_date)